# 03 - Padrões: variação mensal, rankings e correlações

Primeira busca por padrões, sempre com taxas por 100 mil habitantes (notebook 02). O notebook tem três partes:

1. **Variação mensal**: os eventos sobem ou descem em certos meses?
2. **Rankings**: quais UFs e municípios têm as maiores taxas?
3. **Correlações**:
   - entre eventos: UFs com muito de um evento também têm muito de outro?
   - entre séries mensais: UFs e regiões sobem e descem juntas ao longo dos meses?

A clusterização e a detecção de atípicos ficam para o notebook 04.

Lembretes que valem para todo o notebook:

- Os dados cobrem só 10 meses de um único ano.
- Diferenças podem vir da forma de registrar de cada UF.
- Correlação não é causa.

## 1. Preparação

Escolhemos 10 eventos da abrangência Estadual que, segundo o notebook 01, têm as 27 UFs em todos os 10 meses. Cada evento é analisado separadamente, porque as métricas são diferentes: vítimas em uns, ocorrências em outros.

As cores dos gráficos seguem uma paleta fixa: cada região tem sempre a mesma cor, e as correlações usam vermelho (negativa), cinza (zero) e azul (positiva).

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

conexao = sqlite3.connect(Path("..") / "data" / "db" / "crimes.db")

cor_principal = "#2a78d6"
cores_regiao = {"Norte": "#2a78d6", "Nordeste": "#eb6834", "Centro-Oeste": "#1baf7a", "Sudeste": "#eda100", "Sul": "#e87ba4"}
divergente = LinearSegmentedColormap.from_list("divergente", ["#e34948", "#f0efec", "#2a78d6"])
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e5e5e2", "grid.linewidth": 0.6})

In [ ]:
eventos = [
    "Homicídio doloso",
    "Tentativa de homicídio",
    "Feminicídio",
    "Roubo seguido de morte (latrocínio)",
    "Morte por intervenção de Agente do Estado",
    "Estupro",
    "Estupro de vulnerável",
    "Roubo de veículo",
    "Furto de veículo",
    "Tráfico de drogas",
]
lista_sql = ", ".join(f"'{e}'" for e in eventos)

mensal = pd.read_sql(f'''
    SELECT uf, evento, metrica, mes, SUM(quantidade) AS casos
    FROM ocorrencias
    WHERE abrangencia = 'Estadual' AND evento IN ({lista_sql})
    GROUP BY uf, evento, metrica, mes
''', conexao)
ufs = pd.read_sql("SELECT uf, regiao, populacao FROM ufs", conexao)
mensal = mensal.merge(ufs, on="uf")
print(mensal.shape)
print(mensal.groupby(["evento", "metrica"]).agg(linhas=("uf", "size"), ufs=("uf", "nunique"), meses=("mes", "nunique"), casos=("casos", "sum")))

Cada evento precisa ter 270 linhas (27 UFs × 10 meses). Se faltasse alguma, a comparação entre UFs ficaria injusta.

In [ ]:
contagem = mensal.groupby("evento").size()
print("Eventos completos:", (contagem == 270).sum(), "de", len(eventos))
mensal.head()

## 2. Variação mensal no Brasil

Os meses têm de 28 a 31 dias. Por isso, comparamos **casos por dia** em vez de casos no mês: um fevereiro mais baixo pode ser só um mês mais curto.

Para colocar eventos de tamanhos diferentes na mesma escala, usamos um **índice**: o mês dividido pela média dos 10 meses, vezes 100. Um índice 110 quer dizer 10% acima da média do período.

In [ ]:
dias_no_mes = pd.Series([31, 28, 31, 30, 31, 30, 31, 31, 30, 31], index=range(1, 11), name="dias")
brasil = mensal.groupby(["evento", "mes"], as_index=False)["casos"].sum()
brasil["dias"] = brasil["mes"].map(dias_no_mes)
brasil["casos_dia"] = brasil["casos"] / brasil["dias"]
brasil["indice"] = brasil["casos_dia"] / brasil.groupby("evento")["casos_dia"].transform("mean") * 100
print(brasil.shape)
indice = brasil.pivot(index="evento", columns="mes", values="indice").round(0)
indice

Um evento com poucos casos oscila muito só por acaso. Com cerca de 50 latrocínios por mês, por exemplo, variar 10 para cima ou para baixo é comum.

Para separar oscilação de acaso de variação "de verdade", comparamos dois desvios-padrão dos casos mensais:

- o **observado** nos dados;
- o **esperado só pelo acaso**. Para contagens de eventos raros e independentes, esse desvio é aproximadamente a raiz quadrada da média.

A **razão** entre os dois diz:

- perto de 1: a variação entre meses é do tamanho que o acaso produziria;
- bem acima de 1 (por exemplo, acima de 2): há variação além do acaso, cuja causa este notebook não investiga.

Os casos são padronizados para meses de 30 dias. A comparação é uma aproximação, útil como indicação, e não um teste formal.

In [ ]:
brasil["casos_30d"] = brasil["casos_dia"] * 30
dispersao = brasil.groupby("evento")["casos_30d"].agg(media="mean", desvio_observado="std")
dispersao["desvio_acaso"] = np.sqrt(dispersao["media"])
dispersao["razao"] = dispersao["desvio_observado"] / dispersao["desvio_acaso"]
dispersao.round(1).sort_values("razao", ascending=False)

In [ ]:
fig, eixos = plt.subplots(2, 5, figsize=(16, 5.5), sharex=True, sharey=True)
for ax, evento in zip(eixos.flat, eventos):
    serie = brasil[brasil["evento"] == evento]
    ax.plot(serie["mes"], serie["indice"], color=cor_principal, linewidth=2, marker="o", markersize=4)
    ax.axhline(100, color="#8a8984", linewidth=1)
    ax.set_title(evento if len(evento) < 30 else evento[:28] + "...", fontsize=9)
    ax.set_xticks(range(1, 11))
fig.supylabel("índice (média do período = 100)")
fig.supxlabel("mês de 2026")
fig.suptitle("Casos por dia em relação à média de jan a out, Brasil")
plt.tight_layout()
plt.show()

Agora o homicídio doloso por região, em **taxa diária por 100 milhões de habitantes**. Usamos 100 milhões porque a taxa diária por 100 mil seria um número muito pequeno e difícil de ler. A população é a mesma em todos os meses, então a forma de cada linha é a mesma dos casos por dia. O que a taxa acrescenta é colocar regiões de tamanhos diferentes na mesma escala.

In [ ]:
homicidio_regiao = mensal[mensal["evento"] == "Homicídio doloso"].groupby(["regiao", "mes"], as_index=False)["casos"].sum()
populacao_regiao = ufs.groupby("regiao")["populacao"].sum()
homicidio_regiao["taxa_dia"] = (
    homicidio_regiao["casos"] / homicidio_regiao["mes"].map(dias_no_mes)
    / homicidio_regiao["regiao"].map(populacao_regiao) * 1e8
)
serie_regiao = homicidio_regiao.pivot(index="mes", columns="regiao", values="taxa_dia")[list(cores_regiao)]
serie_regiao.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for regiao, cor in cores_regiao.items():
    ax.plot(serie_regiao.index, serie_regiao[regiao], color=cor, linewidth=2, marker="o", markersize=4, label=regiao)
    ax.annotate(regiao, (10, serie_regiao.loc[10, regiao]), xytext=(6, 0), textcoords="offset points", va="center", fontsize=9)
ax.set_xticks(range(1, 11))
ax.set_xlim(0.7, 11.2)
ax.set_ylim(bottom=0)
ax.set_xlabel("mês de 2026")
ax.set_ylabel("vítimas por dia por 100 milhões de hab.")
ax.set_title("Homicídio doloso por região")
ax.legend(frameon=False, ncol=5, loc="lower center", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Rankings por taxa

### 3.1 UFs

A taxa de cada UF em cada evento, de janeiro a outubro, por 100 mil habitantes. Cada evento tem sua própria unidade: vítimas por 100 mil em uns, ocorrências por 100 mil em outros.

In [ ]:
total_uf = mensal.groupby(["uf", "regiao", "evento"], as_index=False).agg(casos=("casos", "sum"), populacao=("populacao", "first"))
total_uf["taxa"] = total_uf["casos"] / total_uf["populacao"] * 1e5
taxas_uf = total_uf.pivot(index="uf", columns="evento", values="taxa")[eventos]
print(taxas_uf.shape)
taxas_uf.round(2)

In [ ]:
posicoes = taxas_uf.rank(ascending=False, method="min").astype(int)
posicoes.insert(0, "regiao", ufs.set_index("uf")["regiao"])
posicoes.sort_values("Homicídio doloso")

Para comparar eventos com escalas tão diferentes num mesmo gráfico, dividimos a taxa de cada UF pela taxa do Brasil no mesmo evento:

- 1 quer dizer igual ao Brasil;
- 2 quer dizer o dobro;
- 0,5 quer dizer a metade.

A escala de cor é logarítmica, para que "o dobro" e "a metade" fiquem à mesma distância do cinza.

In [ ]:
total_brasil = mensal.groupby("evento")["casos"].sum()
taxa_brasil = total_brasil / ufs["populacao"].sum() * 1e5
relativa = taxas_uf / taxa_brasil[eventos]
ordem_ufs = ufs.assign(ordem=ufs["regiao"].map({r: i for i, r in enumerate(cores_regiao)})).sort_values(["ordem", "uf"])["uf"]
relativa = relativa.loc[ordem_ufs]
print("Taxa do Brasil (por 100 mil, jan-out):")
print(taxa_brasil.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
valores = np.log2(relativa)
imagem = ax.imshow(valores, cmap=divergente, norm=TwoSlopeNorm(vmin=-2, vcenter=0, vmax=2), aspect="auto")
for i in range(relativa.shape[0]):
    for j in range(relativa.shape[1]):
        ax.text(j, i, f"{relativa.iat[i, j]:.1f}", ha="center", va="center", fontsize=7, color="#0b0b0b")
ax.set_xticks(range(len(eventos)))
ax.set_xticklabels([e if len(e) < 25 else e[:23] + "..." for e in eventos], rotation=40, ha="right", fontsize=8)
ax.set_yticks(range(len(relativa)))
ax.set_yticklabels([f"{uf} ({ufs.set_index('uf').loc[uf, 'regiao']})" for uf in relativa.index], fontsize=8)
ax.grid(False)
barra = fig.colorbar(imagem, ax=ax, ticks=[-2, -1, 0, 1, 2], shrink=0.6)
barra.ax.set_yticklabels(["1/4", "1/2", "igual", "2x", "4x"])
ax.set_title("Taxa da UF em relação à taxa do Brasil, jan-out 2026")
plt.tight_layout()
plt.show()

### 3.2 Municípios

No nível municipal, só há homicídio doloso e os outros eventos de vítimas listados no notebook 01. Aqui usamos homicídio doloso e removemos a linha "NÃO INFORMADO", que não tem população.

Primeiro, o ranking sem filtro: ele mostra o problema dos **números pequenos**. Num município de 3 mil habitantes, 2 homicídios já dão uma taxa de 67 por 100 mil, que fica no topo do ranking por puro acaso.

In [ ]:
homicidio_municipio = pd.read_sql('''
    WITH casos AS (
        SELECT uf, municipio, SUM(quantidade) AS vitimas
        FROM ocorrencias
        WHERE evento = 'Homicídio doloso' AND abrangencia = 'Estadual' AND municipio != 'NÃO INFORMADO'
        GROUP BY uf, municipio
    )
    SELECT c.uf, c.municipio, m.regiao, m.populacao, c.vitimas,
           c.vitimas * 100000.0 / m.populacao AS taxa
    FROM casos c
    JOIN municipios m ON m.uf = c.uf AND m.municipio = c.municipio
''', conexao)
print(homicidio_municipio.shape)
print("Municípios com zero vítimas:", (homicidio_municipio["vitimas"] == 0).sum())
homicidio_municipio.sort_values("taxa", ascending=False).head(10).round(1)

Uma forma simples de reduzir esse problema é olhar só para municípios grandes. Usamos o corte de **100 mil habitantes**: é uma escolha nossa, não uma regra. No notebook 04 podemos testar métodos que tratam municípios pequenos sem descartá-los.

In [ ]:
grandes = homicidio_municipio[homicidio_municipio["populacao"] >= 100_000].sort_values("taxa", ascending=False)
print("Municípios com 100 mil habitantes ou mais:", len(grandes))
print("Parcela da população do Brasil:", round(grandes["populacao"].sum() / ufs["populacao"].sum() * 100, 1), "%")
print("Parcela das vítimas:", round(grandes["vitimas"].sum() / homicidio_municipio["vitimas"].sum() * 100, 1), "%")
grandes.head(15).round(1)

In [ ]:
grandes.tail(10).round(1)

In [ ]:
print(grandes.groupby("regiao").agg(municipios=("municipio", "size"), mediana_taxa=("taxa", "median")).round(1).sort_values("mediana_taxa", ascending=False))

## 4. Correlações

A **correlação** é um número de -1 a 1 que mede se duas coisas andam juntas:

- **perto de 1**: quando uma é alta, a outra também tende a ser;
- **perto de -1**: quando uma é alta, a outra tende a ser baixa;
- **perto de 0**: não há relação linear.

Ela não diz que uma coisa causa a outra.

### 4.1 Entre eventos (27 UFs)

Pergunta: UFs com taxa alta de um evento também têm taxa alta de outro?

Usamos a correlação de **Spearman**, que compara as posições no ranking em vez dos valores. Assim, uma UF muito diferente das outras não domina o resultado sozinha.

In [ ]:
correlacao_eventos = taxas_uf.corr(method="spearman")
nomes_curtos = [e if len(e) < 25 else e[:23] + "..." for e in eventos]

fig, ax = plt.subplots(figsize=(9, 7.5))
imagem = ax.imshow(correlacao_eventos, cmap=divergente, vmin=-1, vmax=1)
for i in range(len(eventos)):
    for j in range(len(eventos)):
        ax.text(j, i, f"{correlacao_eventos.iat[i, j]:.2f}", ha="center", va="center", fontsize=7.5)
ax.set_xticks(range(len(eventos)))
ax.set_xticklabels(nomes_curtos, rotation=40, ha="right", fontsize=8)
ax.set_yticks(range(len(eventos)))
ax.set_yticklabels(nomes_curtos, fontsize=8)
ax.grid(False)
fig.colorbar(imagem, ax=ax, shrink=0.7, label="correlação de Spearman")
ax.set_title("Correlação entre taxas de eventos nas 27 UFs")
plt.tight_layout()
plt.show()

In [ ]:
pares = correlacao_eventos.where(np.triu(np.ones(correlacao_eventos.shape, dtype=bool), k=1)).stack().dropna().rename("spearman")
print("Pares de eventos:", len(pares))
print("Mais positivos:")
print(pares.sort_values(ascending=False).head(5).round(2))
print()
print("Mais negativos:")
print(pares.sort_values().head(5).round(2))

### 4.2 Entre séries mensais

Pergunta: quando o homicídio sobe num mês numa região, ele também sobe nas outras?

Usamos a taxa diária de homicídio por mês e a correlação de **Pearson**, que compara os valores diretamente.

Com só **10 meses**, a correlação é muito instável. Como referência: com 10 pontos, uma correlação precisa passar de cerca de **0,63** (em valor absoluto) para ser considerada improvável de ter surgido só por acaso, ao nível usual de 5%.

In [ ]:
correlacao_regioes = serie_regiao.corr()
correlacao_regioes.round(2)

O mesmo para as 27 UFs. Com 27 UFs, há 351 pares. Mesmo sem nenhuma relação real, esperamos que cerca de 5% deles (uns 18 pares) passem de 0,63 por acaso. Por isso, o número de pares acima do corte só é informativo se for bem maior que 18.

In [ ]:
homicidio_uf = mensal[mensal["evento"] == "Homicídio doloso"].copy()
homicidio_uf["taxa_dia"] = homicidio_uf["casos"] / homicidio_uf["mes"].map(dias_no_mes) / homicidio_uf["populacao"] * 1e8
serie_uf = homicidio_uf.pivot(index="mes", columns="uf", values="taxa_dia")[ordem_ufs]
correlacao_ufs = serie_uf.corr()

pares_uf = correlacao_ufs.where(np.triu(np.ones(correlacao_ufs.shape, dtype=bool), k=1)).stack().dropna().rename("pearson")
print("Pares de UFs:", len(pares_uf))
print("Esperado acima de 0,63 em valor absoluto só por acaso: cerca de", round(len(pares_uf) * 0.05))
print("Observado acima de 0,63 em valor absoluto:", (pares_uf.abs() > 0.632).sum())
print("Positivos acima de 0,63:", (pares_uf > 0.632).sum(), "| Negativos abaixo de -0,63:", (pares_uf < -0.632).sum())
print("Média de todas as correlações:", round(pares_uf.mean(), 2))

In [ ]:
regiao_da_uf = ufs.set_index("uf")["regiao"]
mais_fortes = pares_uf.reindex(pares_uf.abs().sort_values(ascending=False).index).head(10)
mais_fortes = mais_fortes.rename_axis(["uf_1", "uf_2"]).reset_index()
mais_fortes["regiao_1"] = mais_fortes["uf_1"].map(regiao_da_uf)
mais_fortes["regiao_2"] = mais_fortes["uf_2"].map(regiao_da_uf)
mais_fortes.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8.5))
imagem = ax.imshow(correlacao_ufs, cmap=divergente, vmin=-1, vmax=1)
ax.set_xticks(range(len(ordem_ufs)))
ax.set_xticklabels(ordem_ufs, rotation=90, fontsize=8)
ax.set_yticks(range(len(ordem_ufs)))
ax.set_yticklabels(ordem_ufs, fontsize=8)
limites = ufs["regiao"].map({r: i for i, r in enumerate(cores_regiao)}).value_counts().sort_index().cumsum().values[:-1] - 0.5
for limite in limites:
    ax.axhline(limite, color="#52514e", linewidth=0.8)
    ax.axvline(limite, color="#52514e", linewidth=0.8)
ax.grid(False)
fig.colorbar(imagem, ax=ax, shrink=0.7, label="correlação de Pearson (10 meses)")
ax.set_title("Correlação das séries mensais de homicídio doloso entre UFs\n(ordenadas por região: Norte, Nordeste, Centro-Oeste, Sudeste, Sul)")
plt.tight_layout()
plt.show()

In [ ]:
conexao.close()

## 5. Resumo

O que os prints acima mostram, para os 10 eventos da abrangência Estadual, de jan a out de 2026:

**Variação mensal (Brasil, casos por dia)**
- Variação maior que o acaso produziria (razão bem acima de 1):
  - roubo de veículo, com razão 10,4. O índice fica entre 110 e 121 de janeiro a abril e entre 87 e 97 de maio a outubro;
  - estupro de vulnerável (5,2), tráfico (4,9) e furto de veículo (4,1).
- Variação próxima do que o acaso produziria (razão perto de 1): feminicídio (1,3) e latrocínio (1,4). São eventos com poucos casos por mês.
- Latrocínio tem um índice de 51 em julho. É um único mês, e este notebook não investiga a causa.
- Com apenas 10 meses de um ano, não dá para separar sazonalidade (padrão que se repete todo ano) de tendência ou de acaso. Para afirmar sazonalidade, seriam necessários mais anos.

**Homicídio doloso por região**
- O Nordeste tem a maior taxa diária em todos os meses.
- Norte e Centro-Oeste terminam outubro acima de onde começaram em janeiro. Sudeste e Sul terminam abaixo.

**Rankings de UFs**
- A posição de uma UF muda bastante de um evento para outro. Por exemplo, RJ e PE ficam bem acima do Brasil em roubo de veículo, e AP e RR bem abaixo.
- Algumas UFs ficam bem acima do Brasil num evento específico: AP (4,6 vezes) em morte por intervenção de agente do Estado, MS (5,6 vezes) em estupro e RR (5,7 vezes) em latrocínio. Em RR, isso são só 10 vítimas, de novo o problema dos números pequenos.
- Isso descreve os registros. Diferenças tão grandes merecem checar também se a forma de registrar difere entre as UFs.

**Rankings de municípios (homicídio doloso)**
- O topo do ranking sem filtro é formado por municípios pequenos. O primeiro é Santo André (PB), com 2.714 habitantes e 7 vítimas.
- 2.367 dos 5.570 municípios não tiveram nenhuma vítima no período.
- Com o corte de 100 mil habitantes, ficam 338 municípios, que somam 58,3% da população e 53,4% das vítimas. Entre eles:
  - Itaperuna (RJ) lidera com 41,9 vítimas por 100 mil;
  - a mediana das taxas é maior no Nordeste (19,2) e no Norte (12,3).

**Correlação entre eventos (27 UFs, Spearman)**
- A mais forte é entre estupro e estupro de vulnerável (0,83).
- Roubo de veículo tem correlação negativa com feminicídio (-0,54) e com estupro (-0,50): UFs com taxa alta de roubo de veículo tendem a ter taxa menor desses eventos.
- Com 27 UFs e 45 pares, correlações moderadas como essas podem surgir por acaso. Elas são pistas para investigar, não conclusões.

**Correlação entre séries mensais (homicídio doloso, Pearson, 10 meses)**
- Entre regiões, Sudeste, Sul e Nordeste andam juntos: Sudeste e Sul têm 0,86, e Sudeste e Nordeste 0,84. O Centro-Oeste anda no sentido oposto a esse grupo.
- Entre UFs, 27 dos 351 pares passam de 0,63 em valor absoluto, contra cerca de 18 esperados só por acaso. A média das correlações é 0,11.
- Há algum sinal de movimento comum entre as UFs, mas fraco. Nenhum par específico deve ser lido como relação real sem mais dados.

Para o notebook 04:
- a clusterização pode usar a tabela de taxas relativas por UF (seção 3.1) como perfil de cada UF;
- municípios pequenos precisam de um tratamento para números pequenos.